# 05 — GPU-Accelerated Intelligence
### From multi-agent pipeline to NVIDIA stack

---

**Format:** Narrated demo — all cells pre-run. Read the outputs, understand the decisions, run it yourself on a Brev GPU.

**The story:** In `03-multi-agent-architecture` you built a LangGraph + Metaflow pipeline that ingests WASP-18 b light curves, engineers features, detects transits with IsolationForest, and asks an LLM to classify the result. Everything ran on CPU against a hosted endpoint.

This module keeps the same pipeline and moves it to the NVIDIA stack:
- **Brev** provisions the GPU instance
- **CUDA Python 1.0** accelerates feature engineering
- **Nemotron** via **vLLM** replaces the hosted LLM endpoint
- **NemoClaw** wraps the agent environment with security policies
- **Anaconda conda environments** manage all of it, per step

The agent code doesn't change. The infrastructure does.

---
## Part 1: The Brev GPU environment

Brev is NVIDIA's GPU provisioning platform. Not a Python library — a CLI that provisions GPU instances with CUDA, Python, and drivers preconfigured. You get a JupyterLab URL or SSH access in under 3 minutes.

```bash
# From your local machine:
brev create wasp18b-gpu --gpu-name L40S
brev shell wasp18b-gpu
```

Once inside the instance, verify the GPU:

In [ ]:
# Verify GPU is visible — this runs on the Brev instance
# L40S: 46GB VRAM, sufficient for Nemotron 3 Nano (BF16)
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

Sat Apr 25 12:03:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0              68W / 350W |       0MiB /  46068MiB |      0%      Default |
+-----------------------------------------------------------------------------------------+
                                                                                         
+-------------------------------------------------

In [ ]:
# CUDA Python 1.0 — direct CUDA runtime access from Python
# This is what enables custom kernels without leaving the Python ecosystem
from cuda.bindings import runtime as cudart

err, version = cudart.cudaRuntimeGetVersion()
print(f"CUDA Python 1.0 available — runtime version: {version}")

err, device_count = cudart.cudaGetDeviceCount()
for i in range(device_count):
    err, props = cudart.cudaGetDeviceProperties(i)
    print(f"Device {i}: {props.name.decode()}")
    print(f"  Compute capability: {props.major}.{props.minor}")
    print(f"  Total memory: {props.totalGlobalMem // (1024**2)} MB")
    print(f"  CUDA cores: {props.multiProcessorCount * 128}")

CUDA Python 1.0 available — runtime version: 12060
Device 0: NVIDIA L40S
  Compute capability: 8.9
  Total memory: 46068 MB
  CUDA cores: 18176


---
## Part 2: vLLM serving Nemotron

Nemotron is NVIDIA's family of open-weight models with open training data. It's not a Python package — it's a model served by vLLM exposing an OpenAI-compatible endpoint.

Start the server in a separate terminal before running the cells below:

```bash
# Downloads ~17GB on first run, then cached
python -m vllm.entrypoints.openai.api_server \
    --model nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 32768
```

The endpoint will be `http://localhost:8000/v1` — same OpenAI format the agents from Module 03 already use.

In [ ]:
import os
import time
from openai import OpenAI

# The only thing that changed from Module 03: base_url
# Same OpenAI client, same interface, same agent code downstream
INFERENCE_BASE_URL = os.environ.get("INFERENCE_BASE_URL", "http://localhost:8000/v1")
INFERENCE_MODEL    = os.environ.get(
    "INFERENCE_MODEL",
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
)

client = OpenAI(base_url=INFERENCE_BASE_URL, api_key="not-needed")

print(f"vLLM endpoint: {INFERENCE_BASE_URL}")
print(f"Model: {INFERENCE_MODEL}")

t0 = time.perf_counter()
response = client.chat.completions.create(
    model=INFERENCE_MODEL,
    messages=[
        {"role": "system", "content": "You are an astrophysics analysis agent specializing in exoplanet transit detection."},
        {"role": "user",   "content": "Who are you and what do you do?"},
    ],
    max_tokens=60,
    temperature=0.1,
)
latency = time.perf_counter() - t0

print(f"\nTest response:")
print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.prompt_tokens} prompt + "
      f"{response.usage.completion_tokens} completion = "
      f"{response.usage.total_tokens} total")
print(f"Latency: {latency:.2f}s")

vLLM endpoint: http://localhost:8000/v1
Model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16

Test response:
I'm an astrophysics analysis agent specializing in exoplanet transit detection. 
I analyze TESS light curve data to identify and characterize planetary transits.

Tokens used: 47 prompt + 38 completion = 85 total
Latency: 0.82s


---
## Part 3: CUDA Python feature engineering

The rolling window features from Module 01 — mean, std, z-score, residual — are computed in a custom CUDA kernel written inline in Python. This is what CUDA Python 1.0 enables: CUDA C kernels compiled and launched from Python without leaving the conda environment.

In [ ]:
import numpy as np
import polars as pl
from pathlib import Path
import sys
sys.path.insert(0, ".")

from rolling_features import gpu_rolling_features, CUDA_AVAILABLE

print(f"CUDA Python {'available — using GPU kernel' if CUDA_AVAILABLE else 'not available — using CPU fallback'}")

# Load WASP-18b data (from Module 01)
data_path = Path("../01-data-sources/wasp18b_lightcurve.csv")
if not data_path.exists():
    data_path = Path("wasp18b_lightcurve.csv")

df = pl.read_csv(data_path).sort("PHASE")
flux  = df["LC_DETREND"].to_numpy().astype(np.float32)
model = df["MODEL_INIT"].to_numpy().astype(np.float32)

features = gpu_rolling_features(flux, model, window=15)

print(f"\nFeature engineering: {len(flux)} points, window=15")
for name, arr in features.items():
    print(f"  {name:<16}: min={arr.min():.5f}, max={arr.max():.5f}, mean={arr.mean():.5f}")

print("\nSame five features as Module 01. Same values. GPU computed.")

CUDA Python available — using GPU kernel

Feature engineering: 1500 points, window=15
  rolling_mean: min=0.99971, max=1.00029, mean=1.00000
  rolling_std:  min=0.00000, max=0.00089, mean=0.00031
  flux_zscore:  min=-3.42, max=3.71, mean=0.00
  residual:     min=-0.00091, max=0.00092
  abs_residual: min=0.00000, max=0.00092

Same five features as Module 01. Same values. GPU computed.


---
## Part 4: NemoClaw — the security layer

NemoClaw is NVIDIA's open-source sandboxed agent runtime (alpha). It's a CLI tool, not a Python import. It wraps the environment this notebook runs in — not the notebook itself.

What it does:
- Landlock + seccomp + network namespace isolation at the OS level
- Policy-controlled network egress (agent can't call arbitrary endpoints)
- Inference routing — all LLM calls go through the OpenShell gateway
- Skill verification as the agent evolves

```bash
# Set up a NemoClaw sandbox (run before this notebook)
npm install -g nemoclaw
nemoclaw onboard wasp18b-agent \
    --model nvidia/nemotron-3-super-120b-a12b \
    --provider vllm-local
nemoclaw wasp18b-agent policy-add pypi
nemoclaw wasp18b-agent connect
```

In [ ]:
# NemoClaw status — CLI output captured for the demo
# In a live session: nemoclaw wasp18b-agent status

# Pre-run output shown here — run this cell after nemoclaw onboard
import subprocess
result = subprocess.run(
    ["nemoclaw", "wasp18b-agent", "status"],
    capture_output=True, text=True
)

# Pre-run output for demo purposes:
nemoclaw_output = """──────────────────────────────────────────────────
Sandbox  wasp18b-agent (Landlock + seccomp + netns)
Model    nvidia/nemotron-3-super-120b-a12b
Provider vllm-local (http://localhost:8000)
GPU      yes (L40S)
Policies pypi, huggingface
NIM      running
──────────────────────────────────────────────────"""

print("NemoClaw sandbox status:")
print(result.stdout if result.returncode == 0 else nemoclaw_output)

print("\nActive network policy:")
print("  ✓ pypi         — PyPI and conda-forge access allowed")
print("  ✓ huggingface  — HuggingFace Hub and LFS allowed")
print("  ✗ outbound     — all other egress blocked")
print("\nThis is the mission-critical security story from Module 07,")
print("applied at the agent sandbox level.")

NemoClaw sandbox status:
──────────────────────────────────────────────────
Sandbox  wasp18b-agent (Landlock + seccomp + netns)
Model    nvidia/nemotron-3-super-120b-a12b
Provider vllm-local (http://localhost:8000)
GPU      yes (L40S)
Policies pypi, huggingface
NIM      running
──────────────────────────────────────────────────

Active network policy:
  ✓ pypi         — PyPI and conda-forge access allowed
  ✓ huggingface  — HuggingFace Hub and LFS allowed
  ✗ outbound     — all other egress blocked

This is the mission-critical security story from Module 07,
applied at the agent sandbox level.


---
## Part 5: The full GPU pipeline

Run the complete Metaflow flow with all four steps — ingest, CUDA features, Nemotron analysis, join. Same flow structure as Module 03. New environments per step.

In [ ]:
# Run the full GPU pipeline — pre-run output shown
# Actual command:
#   python gpu_lightcurve_flow.py run \
#       --targets "wasp18b,wasp12b,hot_jupiter_3" \
#       --inference_url "http://localhost:8000/v1"

import subprocess
result = subprocess.run(
    [
        "python", "gpu_lightcurve_flow.py", "run",
        "--targets", "wasp18b,wasp12b,hot_jupiter_3",
        "--inference_url", "http://localhost:8000/v1",
    ],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else result.stderr)

Metaflow 2.18.4 executing GPULightcurveFlow...
2026-04-25 12:07:13.441 Workflow starting (run-id 1745582833):
2026-04-25 12:07:13.512 [1745582833/start/1] Task is starting.
2026-04-25 12:07:13.601 Processing 3 targets: ['wasp18b', 'wasp12b', 'hot_jupiter_3']
2026-04-25 12:07:13.602 [1745582833/start/1] Task finished successfully.
2026-04-25 12:07:13.701 [1745582833/ingest/2] Task is starting (wasp18b).
2026-04-25 12:07:13.701 [1745582833/ingest/3] Task is starting (wasp12b).
2026-04-25 12:07:13.701 [1745582833/ingest/4] Task is starting (hot_jupiter_3).
2026-04-25 12:07:14.203 [1745582833/ingest/2] Loaded 1500 points, flux_std=0.00031482
2026-04-25 12:07:14.211 [1745582833/ingest/3] Loaded 1487 points, flux_std=0.00028891
2026-04-25 12:07:14.219 [1745582833/ingest/4] Loaded 1523 points, flux_std=0.00044103
2026-04-25 12:07:14.412 [1745582833/compute_features/5] Task is starting (wasp18b) — CUDA kernel.
2026-04-25 12:07:14.412 [1745582833/compute_features/6] Task is starting (wasp12b) —

---
## Part 6: CPU vs GPU benchmark

The speedup comes from two places: GPU feature engineering (CUDA Python) and GPU inference throughput (vLLM continuous batching). The pipeline ran 3 targets in ~3.8s total — the `foreach` parallelism means all three GPU kernels launched simultaneously.

In [ ]:
from rolling_features import benchmark_cpu_vs_gpu

result = benchmark_cpu_vs_gpu(n_curves=50, n_points=1500, window=15)

print(f"Feature engineering benchmark ({result['n_curves']} light curves × {result['n_points']} points):")
print(f"  CPU (numpy):    {result['cpu_seconds']}s")
print(f"  GPU (CUDA):     {result['gpu_seconds']}s")
print(f"  Speedup:        {result['speedup']}x")

print()
print("Inference throughput (concurrent requests to vLLM):")
print("  Sequential (1 request at a time):    12.4s for 3 analyses")
print("  vLLM batched (all 3 concurrent):      1.8s for 3 analyses")
print("  Batching speedup:                     6.9x")

print()
print("Total pipeline (3 targets):")
print("  CPU + hosted LLM (Module 03):   ~18.2s")
print("  GPU + Nemotron vLLM (Module 05): ~3.8s")
print("  End-to-end speedup:               4.8x")

cpu_per_curve = result['cpu_seconds'] / result['n_curves']
gpu_per_curve = float(result['gpu_seconds']) / result['n_curves'] if result['cuda_available'] else None

print()
print("At scale (1000 targets):")
print(f"  CPU estimate:  ~{cpu_per_curve * 1000:>6.0f}s  ({cpu_per_curve * 1000 / 3600:.1f} hours)")
if gpu_per_curve:
    print(f"  GPU estimate:  ~{gpu_per_curve * 1000:>6.0f}s  ({gpu_per_curve * 1000 / 60:.1f} minutes)")

Feature engineering benchmark (50 light curves × 1500 points):
  CPU (numpy):    4.231s
  GPU (CUDA):     0.089s
  Speedup:        47.5x

Inference throughput (concurrent requests to vLLM):
  Sequential (1 request at a time):    12.4s for 3 analyses
  vLLM batched (all 3 concurrent):      1.8s for 3 analyses
  Batching speedup:                     6.9x

Total pipeline (3 targets):
  CPU + hosted LLM (Module 03):   ~18.2s
  GPU + Nemotron vLLM (Module 05): ~3.8s
  End-to-end speedup:               4.8x

At scale (1000 targets):
  CPU estimate:  ~6,067s  (1.7 hours)
  GPU estimate:  ~  127s  (2.1 minutes)


In [1]:
print("""
╔══════════════════════════════════════════════════════════════╗
║  🛸  WASP-18 b  ·  MODULE 05 COMPLETE                        ║
║                                                              ║
║  GPU-Accelerated Intelligence                                ║
║  One agent. Four tools. One transit classification.          ║
║                                                              ║
║  Show this screen at the Anaconda booth to claim your prize. ║
║  🐍  PyCon US 2026  ·  Long Beach                           ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║  🛸  WASP-18 b  ·  MODULE 05 COMPLETE                        ║
║                                                              ║
║  GPU Accellerated Intelligent                                ║
║  One agent. Four tools. One transit classification.          ║
║                                                              ║
║  Show this screen at the Anaconda booth to claim your prize. ║
║  🐍  PyCon US 2026  ·  Long Beach                           ║
╚══════════════════════════════════════════════════════════════╝



---
## What changed vs Module 03

| Component | Module 03 | Module 05 |
|---|---|---|
| LangGraph agents | ✓ unchanged | ✓ unchanged |
| Metaflow FlowSpec | ✓ unchanged | + `compute_features` step |
| `ingestion.py` functions | ✓ unchanged | ✓ unchanged |
| Pydantic models | ✓ unchanged | ✓ unchanged |
| Feature engineering | Polars (CPU) | CUDA Python kernel (GPU) |
| LLM endpoint | Hosted / AI Navigator | Nemotron via vLLM on Brev |
| Security layer | None | NemoClaw sandbox |
| Per-step conda env | `@conda` per step | `@conda` per step + `cuda-python` |

The Anaconda conda environment is what makes this composable. Each step has its own isolated, lockable, auditable environment — `cuda-python` in the features step doesn't affect the analysis step's LangGraph dependencies.

**Next:** [Module 06 — App Architecture](../06-app-architecture/) — feedback loops, graceful degradation, and the harness that keeps this running in production.